In [1]:
import pandas as pd
import polars as pl
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.1f}'.format)

## Regional Emissions
Only includes light, medium, and heavy vehicles (bus vehicles are excluded). TNC emissions can be included, depending on model settings reported below.

In [2]:
print(f"Include TNC & Taxi Emissions: {util.input_config['include_tnc_emissions']}")

Include TNC & Taxi Emissions: True


In [3]:
emissions_summary = pd.read_csv(util.output_path / 'emissions/emissions_summary.csv')

network = util.process_network_summary()

In [4]:
df_emissions_summary = emissions_summary.copy()

cols_dict = {'pollutant_name': 'Pollutant', 
             'veh_type': 'Vehicle Type',
             'start_tons': 'Start', 
             'intrazonal_tons': 'Intrazonal', 
             'interzonal_tons': 'Interzonal',
             'total_daily_tons': 'Total Daily (Tons)'}
cols = ['Start', 'Intrazonal','Interzonal', 'Total Daily (Tons)']
df_emissions_summary.rename(columns = cols_dict, inplace=True)


In [5]:
df = df_emissions_summary[df_emissions_summary['Vehicle Type'].isin(['light','medium','heavy'])].copy()
df = df.groupby('Pollutant').sum()
df.rename(columns = cols_dict, inplace=True)
df = df.loc[['CO','NOx','PM25 Total','PM10 Total','CO2 Equivalent','VOCs']]

# FIXME line below is failing at 3.11. I dont see a need for it since there are no decimals in the output.
#df = df.applymap(lambda x: x if x > 100 else str(round(x,1)))
df[cols]

,Start,Intrazonal,Interzonal,Total Daily (Tons)
Pollutant,,,,
CO,48.5,0.9,109.1,158.5
NOx,3.5,0.0,8.6,12.1
PM25 Total,0.3,0.0,0.8,1.0
PM10 Total,0.3,0.1,4.7,5.1
CO2 Equivalent,"1,076.9",134.2,"20,962.5","22,173.6"
VOCs,2.9,0.0,1.1,4.0


## Results by Vehicle Type

### VMT

In [6]:
df_network = network.copy()

df_network['@lveh'] = df_network[['@hov2_inc1','@hov2_inc2', '@hov2_inc3', 
                                  '@hov3_inc1', '@hov3_inc2', '@hov3_inc3',
                                  '@sov_inc1', '@sov_inc2', '@sov_inc3', 
                                  '@tnc_inc1', '@tnc_inc2','@tnc_inc3']].sum(axis=1)

df_network['light'] = df_network['@lveh']*df_network['length']
df_network['medium'] = df_network['@mveh']*df_network['length']
df_network['heavy'] = df_network['@hveh']*df_network['length']

index_labels = ['light','medium','heavy']
df = pd.DataFrame(index=index_labels)
df['VMT'] = df_network[index_labels].sum()

df.index.name = 'Vehicle Type'
df

,VMT
Vehicle Type,
light,"83,555,804.4"
medium,"3,491,660.5"
heavy,"3,064,990.5"


### Emissions

In [7]:
# Calculate emissions and VMT by vehicle type and save results
# Note that Total VMT will not match regional totals because we are not included buses in the emissions summaries

df = df_emissions_summary.copy()
df = df.groupby(['Pollutant','Vehicle Type']).sum()
df.rename(columns = cols_dict, inplace=True)

df.loc[['CO','NOx','PM25 Total','PM10 Total','CO2 Equivalent','VOCs']][cols].copy()


Start  Intrazonal  Interzonal  Total Daily (Tons)
Pollutant      Vehicle Type                                                   
CO             heavy           0.0         0.0         4.8                 4.9
               light          45.2         0.9       100.3               146.4
               medium          3.3         0.0         3.9                 7.2
               transit         0.0         0.0         1.4                 1.4
NOx            heavy           0.0         0.0         5.8                 5.9
               light           3.0         0.0         2.2                 5.1
               medium          0.5         0.0         0.6                 1.1
               transit         0.0         0.0         0.2                 0.2
PM25 Total     heavy           0.0         0.0         0.1                 0.1
               light           0.2         0.0         0.6                 0.9
               medium          0.0         0.0         0.0                 0.1
               transit         0.0         0.0         0.0                 0.0
PM10 Total     heavy           0.0         0.0         0.5                 0.5
               light           0.3         0.1         4.0                 4.3
               medium          0.0         0.0         0.2                 0.3
               transit         0.0         0.0         0.0                 0.0
CO2 Equivalent heavy           2.3         2.7     4,028.5             4,033.5
               light         993.8       130.2    15,448.5            16,572.4
               medium         80.8         1.3     1,485.5             1,567.6
               transit         0.9         0.0       319.8               320.7
VOCs           heavy           0.0         0.0         0.1                 0.1
               light           2.6         0.0         0.9                 3.5
               medium          0.3         0.0         0.1                 0.4
               transit         0.0         0.0         0.0                 0.0